Szia!

Ebben a Colab notebook fájlban készítettük el a harmadik házi feladatot.  

Tagok:

*   Kedves Anita [#KdvsAnita](https://github.com/KdvsAnita) - G02
*   Kozári Márton [#komarc](https://github.com/komarc) - G02
*   Simon Patrik [#polini46corvinus](https://github.com/polini46corvinus) - G04

**LLaMa csapat 🦙**

---

In [2]:
# Alap csomagok importálása
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
# **Adatfeldolgozás**
---

In [3]:
# Nyers fájl betöltése
data = pd.read_csv("/content/drive/My Drive/corvinus_mi2026-g07.csv")
data.info()
print(data.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19382 entries, 0 to 19381
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   #                         19382 non-null  int64  
 1   claim_status              19084 non-null  object 
 2   video_id                  19382 non-null  int64  
 3   video_duration_sec        19382 non-null  int64  
 4   video_transcription_text  19084 non-null  object 
 5   verified_status           19382 non-null  object 
 6   author_ban_status         19382 non-null  object 
 7   video_view_count          19084 non-null  float64
 8   video_like_count          19084 non-null  float64
 9   video_share_count         19084 non-null  float64
 10  video_download_count      19084 non-null  float64
 11  video_comment_count       19084 non-null  float64
dtypes: float64(5), int64(3), object(4)
memory usage: 1.8+ MB
(19382, 12)


In [4]:
# Listázunk minden olyan sort, ahol bármelyik cella üres, hogy megtudjuk, mi hiányzik
data[data.isnull().any(axis="columns")]

,#,claim_status,video_id,video_duration_sec,video_transcription_text,verified_status,author_ban_status,video_view_count,video_like_count,video_share_count,video_download_count,video_comment_count
19084,19085,NaN,4380513697,39,NaN,not verified,active,NaN,NaN,NaN,NaN,NaN
19085,19086,NaN,8352130892,60,NaN,not verified,active,NaN,NaN,NaN,NaN,NaN
19086,19087,NaN,4443076562,25,NaN,not verified,active,NaN,NaN,NaN,NaN,NaN
19087,19088,NaN,8328300333,7,NaN,not verified,active,NaN,NaN,NaN,NaN,NaN
19088,19089,NaN,3968729520,8,NaN,not verified,active,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
19377,19378,NaN,7578226840,21,NaN,not verified,active,NaN,NaN,NaN,NaN,NaN
19378,19379,NaN,6079236179,53,NaN,not verified,active,NaN,NaN,NaN,NaN,NaN
19379,19380,NaN,2565539685,10,NaN,verified,under review,NaN,NaN,NaN,NaN,NaN
19380,19381,NaN,2969178540,24,NaN,not verified,active,NaN,NaN,NaN,NaN,NaN


In [5]:
# Az utolsó 298 sorban konzisztensen hiányzik sok oszlop értéke, a megmaradt értékek felhasználásával pedig már nem tudunk hasznos információhoz jutni, ezért töröljük ezeket.
data = data.dropna()
data.shape

(19084, 12)

In [6]:
# Az első oszlop ("#") vélhetően csak egy index, nem pedig egy azonosításra is alkalmas sorszám. Ellenőrizzük, hogy szigorúan monoton növekvő-e, és ha igen, akkor törölhető a "#" oszlop.
idx = data.index
val = 1
for i in range(len(idx)-1):
    if idx[i+1]>idx[i]:
      val+=1

print(f"Mivel {val} értéket kaptunk, így az azonosítóoszlop törölhető.")
data = data.set_index("#")
data.index.name=None
print(data.index)
data

Mivel 19084 értéket kaptunk, így az azonosítóoszlop törölhető.
Index([    1,     2,     3,     4,     5,     6,     7,     8,     9,    10,
       ...
       19075, 19076, 19077, 19078, 19079, 19080, 19081, 19082, 19083, 19084],
      dtype='int64', length=19084)


,claim_status,video_id,video_duration_sec,video_transcription_text,verified_status,author_ban_status,video_view_count,video_like_count,video_share_count,video_download_count,video_comment_count
1,claim,7017666017,59,someone shared with me that drone deliveries a...,not verified,under review,343296.0,19425.0,241.0,1.0,0.0
2,claim,4014381136,32,someone shared with me that there are more mic...,not verified,active,140877.0,77355.0,19034.0,1161.0,684.0
3,claim,9859838091,31,someone shared with me that american industria...,not verified,active,902185.0,97690.0,2858.0,833.0,329.0
4,claim,1866847991,25,someone shared with me that the metro of st. p...,not verified,active,437506.0,239954.0,34812.0,1234.0,584.0
5,claim,7105231098,19,someone shared with me that the number of busi...,not verified,active,56167.0,34987.0,4110.0,547.0,152.0
...,...,...,...,...,...,...,...,...,...,...,...
19080,opinion,1492320297,49,in our opinion the earth holds about 11 quinti...,not verified,active,6067.0,423.0,81.0,8.0,2.0
19081,opinion,9841347807,23,in our opinion the queens in ant colonies live...,not verified,active,2973.0,820.0,70.0,3.0,0.0
19082,opinion,8024379946,50,in our opinion the moon is moving away from th...,not verified,active,734.0,102.0,7.0,2.0,1.0
19083,opinion,7425795014,8,in our opinion lightning strikes somewhere on ...,not verified,active,3394.0,655.0,123.0,11.0,4.0


In [7]:
# Az utolsó öt oszlop a fejlécek nevéből ítélve logikusan csak egész számokból állhatna, mégis lebegőpontos értékként tárolja az adatbázis.
# Ellenőrizzük, hogy az utolsó öt oszlop biztonságosan átalakíthatjuk-e egész számokká?
(data.iloc[:,-5:]%1).sum()

,0
video_view_count,0.0
video_like_count,0.0
video_share_count,0.0
video_download_count,0.0
video_comment_count,0.0


In [8]:
# Mivel azt látjuk, hogy az összes oszlop 1-gyel való maradékos osztásának összege nulla, így átalakíthatjuk egésszé.
# A "video_id" oszlop valójában nem számértékű, hanem kategorikus változó, de az egész számok string-ként való tárolása fölösleges, így ezt meghagyjuk egészként.
for i in data.iloc[:,-5:].columns:
  data[i] = data[i].astype(int)

data.info()
data.head()

<class 'pandas.core.frame.DataFrame'>
Index: 19084 entries, 1 to 19084
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   claim_status              19084 non-null  object
 1   video_id                  19084 non-null  int64 
 2   video_duration_sec        19084 non-null  int64 
 3   video_transcription_text  19084 non-null  object
 4   verified_status           19084 non-null  object
 5   author_ban_status         19084 non-null  object
 6   video_view_count          19084 non-null  int64 
 7   video_like_count          19084 non-null  int64 
 8   video_share_count         19084 non-null  int64 
 9   video_download_count      19084 non-null  int64 
 10  video_comment_count       19084 non-null  int64 
dtypes: int64(7), object(4)
memory usage: 1.7+ MB


,claim_status,video_id,video_duration_sec,video_transcription_text,verified_status,author_ban_status,video_view_count,video_like_count,video_share_count,video_download_count,video_comment_count
1,claim,7017666017,59,someone shared with me that drone deliveries a...,not verified,under review,343296,19425,241,1,0
2,claim,4014381136,32,someone shared with me that there are more mic...,not verified,active,140877,77355,19034,1161,684
3,claim,9859838091,31,someone shared with me that american industria...,not verified,active,902185,97690,2858,833,329
4,claim,1866847991,25,someone shared with me that the metro of st. p...,not verified,active,437506,239954,34812,1234,584
5,claim,7105231098,19,someone shared with me that the number of busi...,not verified,active,56167,34987,4110,547,152


In [9]:
# Észrevettem, hogy a videók leírásaiban sokszor szerepel dupla szóköz, ami tokenizálásnál gondot okozhat.
print("Előtte:", len(data[data["video_transcription_text"].str.contains("  ")]))

# Helyettesítsük a dupla szóközöket egy szóközzel.
data["video_transcription_text"] = data["video_transcription_text"].str.replace("  ", " ")
print("Utána:", len(data[data["video_transcription_text"].str.contains("  ")]))

Előtte: 6236
Utána: 0


In [10]:
print(f"Bár {len(data)} adatunk van, valójában csak {data["video_transcription_text"].nunique()} db egyedi leírással találkozunk.")

# Kilistázzuk az egynél többször előforduló leírásokat.
print("Ezek az értékek legalább kétszer szerepelnek:")
data["video_transcription_text"].value_counts().sort_values(ascending=False)[data["video_transcription_text"].value_counts().sort_values(ascending=False)>1]

Bár 19084 adatunk van, valójában csak 19012 db egyedi leírással találkozunk.
Ezek az értékek legalább kétszer szerepelnek:


,count
video_transcription_text,
i read in the media a claim that australia has the most number of beaches in the world,2
someone read in the media that armadillos have bulletproof shells,2
someone read in the media that it is physically impossible for humans to keep their eyes open while sneezing,2
someone read in the media a claim that the longest recorded cricket match lasted 14 days,2
a colleague read in the media a claim that it snows metal and rains acid on venus,2
...,...
a friend read in the media a claim that marie curie is the only person with two nobel prizes in different sciences,2
i learned from the media a claim that penguins can jump as high as 9 feet in the air,2
a friend read in the media a claim that the longest location name on the planet earth is 85 letters in english.,2


In [11]:
# Feltűnt, hogy a videó leírások első néhány kezdőszava sok helyen megegyezik.
# Ha az első három szóra szűrünk, akkor ezekkel a "prefixumokkal", és azok gyakoriságaival találkozunk.
data["video_transcription_text"].str.split().str[:3].value_counts()

,count
video_transcription_text,
"[a, colleague, read]",843
"[a, friend, read]",833
"[a, friend, learned]",757
"[a, colleague, learned]",736
"[a, colleague, discovered]",386
...,...
"[a, friend, mentioned]",76
"[someone, read, online]",71
"[i, claimed, an]",69


In [12]:
data.head()

,claim_status,video_id,video_duration_sec,video_transcription_text,verified_status,author_ban_status,video_view_count,video_like_count,video_share_count,video_download_count,video_comment_count
1,claim,7017666017,59,someone shared with me that drone deliveries a...,not verified,under review,343296,19425,241,1,0
2,claim,4014381136,32,someone shared with me that there are more mic...,not verified,active,140877,77355,19034,1161,684
3,claim,9859838091,31,someone shared with me that american industria...,not verified,active,902185,97690,2858,833,329
4,claim,1866847991,25,someone shared with me that the metro of st. p...,not verified,active,437506,239954,34812,1234,584
5,claim,7105231098,19,someone shared with me that the number of busi...,not verified,active,56167,34987,4110,547,152


In [13]:
# A videó leírás oszlop kívételével a többi kategorikus oszlop jól kódolható, mert mindegyik csak két-három egyedi értéket tartalmaz.
print(data["claim_status"].value_counts())
print("\n")
print(data["verified_status"].value_counts())
print("\n")
print(data["author_ban_status"].value_counts())

convertables = ["claim_status", "verified_status", "author_ban_status"]
num_ohc_cols = data[convertables].apply(lambda x: x.nunique()).sort_values(ascending=False)
num_ohc_cols.name = "egyedi értékek száma kategóriánként"
num_ohc_cols

claim_status
claim      9608
opinion    9476
Name: count, dtype: int64


verified_status
not verified    17884
verified         1200
Name: count, dtype: int64


author_ban_status
active          15383
under review     2066
banned           1635
Name: count, dtype: int64


,egyedi értékek száma kategóriánként
author_ban_status,3
claim_status,2
verified_status,2


In [14]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

data_ohc = data.copy()
le = LabelEncoder()
ohc = OneHotEncoder(drop="first")
enc = ohc.fit_transform(data[convertables])

for col in data[convertables]:
  dat = le.fit_transform(data_ohc[col]).astype(int)
  data_ohc = data_ohc.drop(col, axis=1)
  new_dat = ohc.fit_transform(dat.reshape(-1,1)).astype(int)
  n_cols = new_dat.shape[1]
  col_names = ['_'.join([col, str(x)]) for x in range(n_cols)]
  new_df = pd.DataFrame(new_dat.toarray(), index=data_ohc.index, columns=col_names)
  data_ohc = pd.concat([data_ohc, new_df], axis=1)


print("Oszlopok kódolása:")

print(data_ohc["claim_status_0"].value_counts())       # 0: claim, 1: opinion
print("\n")
print(data_ohc["verified_status_0"].value_counts())    # 0: not verified, 1: verified
print("\n")
print(data_ohc["author_ban_status_0"].value_counts())  # 0: active/under review, 1: banned
print("\n")
print(data_ohc["author_ban_status_1"].value_counts())  # 0: active/banned, 1: under review

Oszlopok kódolása:
claim_status_0
0    9608
1    9476
Name: count, dtype: int64


verified_status_0
0    17884
1     1200
Name: count, dtype: int64


author_ban_status_0
0    17449
1     1635
Name: count, dtype: int64


author_ban_status_1
0    17018
1     2066
Name: count, dtype: int64


In [15]:
#data_ohc[(data_ohc["author_ban_status_0"]==0) & (data_ohc["author_ban_status_1"]==0)] # Így kell az active rendezőket kilistázni.

Bár a hiányzó adatokat már kidobtuk, a valós adathalmazokban gyakoriak a technikai hibák. Például logikailag lehetetlen, hogy egy videónak több like-ja, megosztása vagy kommentje legyen, mint ahányan egyáltalán megnézték. Az ilyen sorok összezavarhatják a korreláció-számítást és a gépi tanulást.

In [16]:
# Ellenőrizzük, vannak-e olyan videók, ahol több a like vagy a komment, mint a megtekintés
anomalies = data_ohc[(data_ohc['video_like_count'] > data_ohc['video_view_count']) |
                     (data_ohc['video_comment_count'] > data_ohc['video_view_count'])]

print(f"Logikai anomáliák száma: {len(anomalies)}")

# Ha vannak ilyenek, érdemes kidobni őket:
if len(anomalies) > 0:
    data_ohc = data_ohc.drop(anomalies.index)

Logikai anomáliák száma: 0


El kell távolítani a felesleges írásjeleket és a stop-szavakat (kötőszavak, névelők: "the", "a", "is", "with"), mert ezek nem hordoznak jelentést, csak zajt a modellnek

In [18]:
import spacy

# 1. Angol nyelvi modell betöltése
nlp_en = spacy.load("en_core_web_sm")

# 2. Szövegtisztító és lemmatizáló függvény definiálása a spaCy segítségével
def clean_and_lemmatize_with_spacy(text):
    # A szöveg átalakítása spaCy Doc objektummá
    doc = nlp_en(str(text).lower())

    cleaned_tokens = []
    for token in doc:
        # Kiszűri a stop szavakat (is_stop) és az írásjeleket/szimbólumokat (PUNCT, SYM)
        if not token.is_stop and token.pos_ not in ['PUNCT', 'SYM']:
            # Számok eltüntetése (opcionális, de a füzetedben is szerepel az is_digit)
            if not token.is_digit:
                # A szótári alakot (lemma_) menti el a nyers szöveg helyett
                cleaned_tokens.append(token.lemma_)

    # A tisztított tokenekből újra egy mondatot formál
    return " ".join(cleaned_tokens)

# 3. Alkalmazzuk az adatbázis 'video_transcription_text' oszlopára
data_ohc['nlp_ready_text'] = data_ohc['video_transcription_text'].apply(clean_and_lemmatize_with_spacy)

# Eredmény ellenőrzése
print(data_ohc[['video_transcription_text', 'nlp_ready_text']].head())

                            video_transcription_text  \
1  someone shared with me that drone deliveries a...   
2  someone shared with me that there are more mic...   
3  someone shared with me that american industria...   
4  someone shared with me that the metro of st. p...   
5  someone shared with me that the number of busi...   

                                      nlp_ready_text  
1                 share drone delivery happen common  
2    share microorganism teaspoon soil people planet  
3  share american industrialist andrew carnegie n...  
4  share metro st . petersburg average depth mete...  
5  share number business allow employee bring pet...  
